# M3L4 E00 — Logs tradicionales vs Tracing estructurado
### Módulo 3 · Lecture 4 · Construcción, pruebas y trazabilidad de agentes en producción

---

## Qué necesitas saber antes

Este ejercicio asume que ya viste:

| Módulo | Concepto | Por qué lo necesitás acá |
|---|---|---|
| M3L2 | Agentes ReAct con LangChain | El tracing registra cada paso Thought-Action-Observation |
| M3L3 | Sistemas multi-agente | Un trace típico tiene spans de routing + agente + herramienta |
| M3L3 | Observabilidad básica | Logs tradicionales como forma inicial de monitoreo |
| M3L3 | Debugging con prints | Los `print()` son la versión más básica de log |

Si alguno de estos conceptos no está claro, repasalos antes de continuar.

---

## Definiciones clave

| Concepto | Definición simple | Cómo aparece en este notebook |
|---|---|---|
| **Log tradicional** | Evento de texto suelto sin conexión entre eventos | `print('INFO: request received')` |
| **Trace** | Registro completo del recorrido de una request, con jerarquía y contexto | Diccionario `trace` con spans, input, output |
| **Span** | Un paso lógico dentro de un trace (una acción, un agente, una tool) | Cada item en `trace['spans']` con nombre, duración, input/output |
| **Jerarquía** | Relación padre-hijo entre spans: un trace contiene varios spans | Trace -> Span: estructura de árbol |
| **metadata** | Información adicional que enriquece el trace (environment, user_id, intent esperado) | `trace['metadata']['expected_intent']` vs `actual_intent` |
| **Misclassification** | Error donde el sistema clasifica un intent incorrectamente | Trace con `expected_intent != actual_intent` en metadata |
| **duration_ms** | Duración de un paso o del trace completo en milisegundos | `'duration_ms': 120` en cada span |

---

## Cómo encaja esto en un sistema de agentes

```
Request del usuario
    |
    v
Sistema de agentes (M3L1-M3L3)
    |  Cada paso: Thought -> Action -> Observation
    v
Registro tradicional (logs)
    |  print(), logging.info() -> eventos sueltos, sin contexto
    v
Tracing estructurado (M3L4 E00)
    |  Trace completa con spans, duración, jerarquía
    v
Diagnóstico y mejora continua (E03, E11, E12)
```

**Objetivo del ejercicio:** entender por qué los logs tradicionales no alcanzan para debuggear sistemas de agentes y qué ventajas ofrece el tracing estructurado.

## Instalación e imports

Este notebook no requiere librerías externas. Todo el código usa tipos nativos de Python:

- `print()` para simular logs tradicionales
- Diccionarios (`{}`) para representar traces y spans
- `time` (no se importa porque no medimos duración real aún, solo la mostramos como valor fijo)

En E01 vas a importar `time`, `uuid` y `datetime` para construir un tracer real. Acá trabajamos con datos estáticos para enfocarnos en el concepto.

## Parte 1 — Logs tradicionales

Así luce el logging típico que todos conocemos.

In [ ]:
def process_request_with_logs(query: str):
    print('INFO: request received')
    print(f'INFO: user query = {query}')
    print('INFO: classified intent')
    print('INFO: agent executed')
    print('INFO: response returned')

process_request_with_logs('No puedo ver mi factura')

## Preguntas para reflexionar

Analiza la salida de la celda anterior y responde:

1. ¿Se puede saber **qué intent** detectó el sistema?
2. ¿Se puede saber **qué agente** respondió?
3. ¿Se puede saber **cuánto tardó** cada paso?
4. ¿Hubo **retrieval vacío**?
5. ¿Hubo **un error** en algún paso intermedio?

> **Tu respuesta aquí:** (doble click para editar)

## Parte 2 — Trace estructurado

Ahora mira cómo se vería la misma request como traza estructurada.

El siguiente diccionario representa la **misma request** que los logs de arriba — pero con toda la información organizada.

In [ ]:
trace = {
    'trace_name': 'support-request',
    'input': 'No puedo ver mi factura',
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': 'No puedo ver mi factura',
            'output': {'intent': 'finance'},
            'duration_ms': 120
        },
        {
            'name': 'finance-agent',
            'input': 'No puedo ver mi factura',
            'output': {'answer': 'Podés ver tu factura desde el portal de pagos.'},
            'duration_ms': 860
        }
    ],
    'output': {
        'final_answer': 'Podés ver tu factura desde el portal de pagos.'
    }
}

trace

## TODO — Completar la comparación

Comparando logs vs trace, completa la siguiente tabla en Markdown:

| Pregunta | ¿Los logs lo responden? | ¿El trace lo responde? |
|---|---|---|
| ¿Qué intent se detectó? | TODO | TODO |
| ¿Qué agente respondió? | TODO | TODO |
| ¿Cuánto tardó cada paso? | TODO | TODO |
| ¿Hubo retrieval vacío? | TODO | TODO |
| ¿Dónde ocurrió el error? | TODO | TODO |

## Parte 3 — Trace con error

Este trace muestra una **misclassification**: el sistema enrutó mal la consulta.

In [ ]:
bad_trace = {
    'trace_name': 'support-request',
    'input': {'query': 'No puedo ver mi factura'},
    'metadata': {
        'expected_intent': 'finance',
        'actual_intent': 'it'
    },
    'spans': [
        {
            'name': 'orchestrator-routing',
            'input': {'query': 'No puedo ver mi factura'},
            'output': {'intent': 'it'},
            'duration_ms': 140
        },
        {
            'name': 'it-agent',
            'input': {'query': 'No puedo ver mi factura'},
            'output': {'response': 'Probá reiniciar la app.'},
            'duration_ms': 900
        }
    ],
    'output': {
        'final_response': 'Probá reiniciar la app.'
    }
}

bad_trace

## TODO — Analizar el error

1. ¿En qué span ocurrió el problema?
2. ¿Cuál era el intent esperado y cuál fue el real?
3. ¿Qué información del trace permite detectar el bug?
4. ¿Qué cambio harías para corregirlo?

> **Tu respuesta aquí:**

## Errores comunes

| Error | Causa | Cómo detectarlo |
|---|---|---|
| Confundir log con trace | Pensar que un `print()` alcanza para debuggear sistemas multi-agente | Los logs no muestran jerarquía ni duración por paso |
| Trace sin metadata | Olvidar incluir `expected_intent`, `user_id` o `environment` | El trace solo muestra datos, no contexto para diagnosticar |
| Ignorar duración | No registrar `duration_ms` en cada span | No se puede detectar latencia alta ni cuellos de botella |
| Span único | Poner todo en un solo span en vez de separar routing, agente, tool | La jerarquía se pierde y no se puede aislar fallas |
| Output sin estructura | Devolver texto plano como output en vez de `{'intent': ..., 'answer': ...}` | El trace se vuelve ilegible para otros sistemas |

## Síntesis

| Logs tradicionales | Tracing estructurado |
|---|---|
| Eventos sueltos, sin contexto | Historia completa de la request |
| No muestran jerarquía | Muestra spans en orden y anidados |
| Sin input/output por paso | Input y output en cada span |
| Difícil de medir latencias | `duration_ms` por span |
| Difícil de detectar misclassifications | `metadata.expected_intent` vs `actual_intent` |
| No se pueden filtrar por etiquetas | `tags` permiten buscar y agrupar traces |
| Se pierden al cerrar la terminal | Quedan almacenados en un sistema de tracing (Langfuse) |

## Relación con otros ejercicios

| Ejercicio | Conexión con E00 |
|---|---|
| **E01** | Vas a construir MiniTracer, un sistema de tracing en Python puro |
| **E02** | Vas a aplicar MiniTracer a un sistema multi-agente real |
| **E03** | Vas a diagnosticar fallas leyendo traces (como el de la Parte 3) |
| **E04 en adelante** | Vas a usar Langfuse, que hace tracing profesional automático |